# Batched sampling with per-request temperature, top-k and top-p

A serving engine samples the next token for a whole batch at once, and every request brings its own sampling settings. A Python loop over requests would stall the GPU, so the filtering has to be vectorised across rows that each use a different temperature, `top_k` and `top_p`.

Write `sample_batch(logits, temperature, top_k, top_p, u)` in PyTorch and return `(tokens, probs)`.

## Inputs
- `logits`: float64 `(B, V)`.
- `temperature`: float64 `(B,)`, values $\ge 0$. `top_k`: int64 `(B,)`, `0` means off. `top_p`: float64 `(B,)` in $(0, 1]$, `1.0` means off.
- `u`: float64 `(B,)` in $[0, 1)$, one uniform draw per row (the randomness is passed in so results are reproducible).

## Per row, in this order
1. **Greedy.** If `temperature == 0`: the token is the index of the largest logit (smallest index on ties) and the row of `probs` is one-hot there. Skip the remaining steps for that row.
2. **Temperature.** $p = \mathrm{softmax}(\text{logits} / T)$.
3. **Top-k** (if `top_k > 0`). Rank tokens by $p$, highest first, ties to the smaller index. Keep the first `top_k`, zero the rest, renormalise.
4. **Top-p** (if `top_p < 1`). In the same ranking, keep the shortest prefix whose cumulative probability is $\ge$ `top_p` (at least one token), zero the rest, renormalise.
5. **Sample.** Return the smallest index $i$, in **vocabulary order**, with $\sum_{j \le i} p_j > u$. If rounding leaves no such index, return the largest index with $p > 0$.

## Output
- `tokens`: int64 `(B,)`.
- `probs`: float64 `(B, V)`, the final distribution of each row (sums to 1, exactly 0 at filtered tokens).

No Python loops over the batch.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def sample_batch(logits, temperature, top_k, top_p, u):
    """Return (tokens (B,) int64, probs (B,V) float64)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[2.0, 1.0, 0.0, -1.0]], "dtype": "float64"}, {"__nd__": [1.0], "dtype": "float64"}, {"__nd__": [2], "dtype": "int64"}, {"__nd__": [1.0], "dtype": "float64"}, {"__nd__": [0.5], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0], "dtype": "int64"}, {"__nd__": [[0.7310585786300049, 0.2689414213699951, 0.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.0, 3.0, 3.0], [1.0, 2.0, 3.0]], "dtype": "float64"}, {"__nd__": [0.0, 1.0], "dtype": "float64"}, {"__nd__": [0, 0], "dtype": "int64"}, {"__nd__": [1.0, 0.5], "dtype": "float64"}, {"__nd__": [0.9, 0.1], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [1, 2], "dtype": "int64"}, {"__nd__": [[0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-3.476532796993764, -2.673285586362265, -2.722213417129974, -0.7032342625568195, -4.625163159393407, -0.37779439216921556, -1.914458456192692, 1.7872003698599577, 1.913694475158468, 2.784516458278173], [1.5349402261894156, -0.10605955751453547, 1.7195879778878191, 3.0109623127676866, -1.307189066834097, 1.220702291660374, -0.08534765542170475, 2.880033450830479, -1.6737900401936867, -0.6030932191310532], [0.7246771863388783, 0.5162205340419951, -3.2788959248952962, 0.720310464474772, -0.23699539903394576, -0.47949569840312406, -0.3106033240045863, 0.43794341015643834, -3.632791322909281, 3.1049331359937327], [-1.7228833465771365, -4.482735716374575, -0.16394898766968208, 2.914960835048829, -1.0372019422064795, 3.1025512418113363, 3.1138840020571537, -1.7254638342227526, -4.930241630554709, -2.4703655136156977]], "dtype": "float64"}, {"__nd__": [0.7, 1.0, 1.3, 0.0], "dtype": "float64"}, {"__nd__": [3, 0, 5, 2], "dtype": "int64"}, {"__nd__": [1.0, 0.8, 0.6, 0.5], "dtype": "float64"}, {"__nd__": [0.9560959106658995, 0.7201014252153678, 0.9887735986731542, 0.11877710929829022], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [9, 7, 9, 6], "dtype": "int64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.15736066327158185, 0.1885279466103662, 0.654111390118052], [0.0, 0.0, 0.12772799414161604, 0.4646466814415308, 0.0, 0.0, 0.0, 0.4076253244168531, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [1.0, 1.0], "dtype": "float64"}, {"__nd__": [2, 0], "dtype": "int64"}, {"__nd__": [1.0, 0.5], "dtype": "float64"}, {"__nd__": [0.99, 0.75], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [1, 2], "dtype": "int64"}, {"__nd__": [[0.5, 0.5, 0.0, 0.0, 0.0], [0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.00018014422029518804, -0.026108483729201715, 0.8720449149097206, 0.9890499704668804, -0.9322490224957446, -0.15675869871669845, -1.1339346465649662, 0.07189067206172675], [-1.1506078219780655, -1.200251555452681, 2.1234219339526375, 0.032388888425520096, 0.6431786123703356, 2.5380983786528484, 0.7858388236905651, -0.11414260193201425], [0.05458995154441833, -0.7444919671299762, 1.094338440936761, -0.3302786854546604, 1.7627455255994957, -0.3447585414426272, -0.3536229319237997, -0.41937740024787623]], "dtype": "float64"}, {"__nd__": [2.0, 0.5, 1.0], "dtype": "float64"}, {"__nd__": [1, 0, 8], "dtype": "int64"}, {"__nd__": [1.0, 0.01, 0.999], "dtype": "float64"}, {"__nd__": [0.3, 0.0, 0.6], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [3, 5, 4], "dtype": "int64"}, {"__nd__": [[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.08041376317497344, 0.03616542046857247, 0.22745077348582357, 0.05472478465180106, 0.4437853143948564, 0.05393808703048085, 0.05346207167522794, 0.050059785118264366]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.8770816392028804, 0.5608082082862007, 4.114297222133552, -1.6600488678783352, 0.8707144336291781, 5.813885203685317, -0.721145060862289, 1.5978383821084132, 4.715646720495352, 1.6603373397309578, -1.903515504318218, -6.187050054497222, 7.179490933499791, 1.2909428463910964, 3.781586662821095, -1.4289406474149229, 5.455445169449424, 1.5586180805791514, 0.14939232994319682, -2.7054515736697136, 1.1045541617448198, -1.4595900972931277, -0.6683075381123484, -4.867532452176848, -1.4716617907124383, 0.4796371045139395, -3.7524184800092915, 0.22168952870011366, 1.9898816368735333, -0.9202798220913573, -1.8623524798788105, -1.8603725628288406, 2.8234269179425047, -4.603515135896732, -0.9013541799317257, -1.258215322455107, 0.43199112737799844, -0.20871026094138379, -3.32251846987229, 6.120268461511428, 0.5328353087637476, -4.026114980924164, -2.6743003495099917, 0.2927546265135015, 1.838909815102379, -4.026465193745677, 2.1340106223028705, -4.119876248731046, -2.383166516110886, 2.5593037877045894], [5.102991687883969, -0.2541236148652626, -0.9286158522171473, 3.803697445918539, -3.2159366607899997, 0.8337654921539777, 0.6997367164712296, 3.027965280632122, 0.8957353119991578, -0.799183003192691, 1.8436637531595563, -3.3796945848906126, 2.442229293850839, -2.0180295276017977, -2.731097703164932, -1.1029391702733102, 0.13800432458311737, 2.1317577316102234, -0.638012021245328, 2.207168554405482, 2.9252474582465156, -0.3172764238320644, -2.944641331591242, 2.7592532676715225, 0.826344742040769, -6.345580605767856, -2.9648659558914465, 0.42948507068472086, -4.115716542895732, -2.044260427380568, -5.138766422293576, 1.20679083668344, 0.7245534202897771, 0.5283768041356984, 4.342422189211496, -1.0578050124236185, -0.8675405563469147, 6.059769846220066, -3.902588919911944, 2.1874887103536196, -3.9452896024848423, 0.7087673676683519, 6.347347875152461, 0.814068074745394, -2.315468316012409, 7.152648177277962, -1.6314641241452572, -0.6185894995105908, 0.9131161806730911, -4.027695592589705], [3.3452948515379486, 4.856475176771853, 2.2762974443964845, 0.9910830806184217, -0.44789306410932983, 2.6602067466687367, -9.450234840571513, -0.37431078659664235, 0.7683771895674514, -5.956590581024219, 1.278502014267122, -0.38251495393885515, -0.10369791491046809, 0.30417370128689525, 1.7254110429916103, 1.3738876269034936, -3.1120559569179056, -0.35088593111950234, -0.5462949751731049, 0.1152673154707226, -1.77327387154523, 2.909561736914143, 3.913645099998197, 4.56509934889443, 5.716956824093286, 4.479841965464767, -1.379794899944565, -1.8431306176693099, -1.1929168066966507, 1.6753156319396567, -0.38949193349829103, 2.973280729642738, 4.951174466361267, -1.2907837189555176, 2.9391231802133744, -1.178834362711785, -3.799492505239085, -3.8065305994166536, -0.9573889434129768, -5.697302825654253, 1.497931450448017, 4.532043013407316, -3.495186322936569, -1.0029422768083207, 4.630674930056153, 1.0416959891718114, 0.586999874231083, -0.7846019588995101, 3.055976383465663, 0.9877269980542703], [-3.2525631662931467, -7.296279906105327, -8.045076892855114, 1.9730143426796474, -0.7982534381993216, 4.344389358382248, 1.5299835987700683, -3.816110985821077, 0.871166799440809, 3.5755847880387246, -2.705270062968918, -1.5262270998279241, 5.222961532383354, -5.239176024086174, -3.2708607400387013, 3.150119601404685, 0.596987207120839, -3.4274725625827216, -4.365736234065356, 1.3903642624556114, 3.642808592471468, 2.479753537564337, 1.2335559742576718, 4.486419706854051, 0.933428394546596, -2.4974317355916047, 6.0956411647526565, -3.702162307772013, -0.4036155547995036, -0.21920194836909795, 4.29340886393673, -2.2911097205464497, -0.8301603659375809, -5.341219250501962, 1.0126044130751646, 1.335487462028179, 1.9520319001699171, -2.7802015062655276, 0.7090878139477578, -2.652434537772221, -1.4373565836394584, -0.23573981812078826, 3.8779394017857918, -1.3208442204476722, 0.5219186870491133, -2.5423027349996454, -1.7198532583836026, -2.1155650316555494, -0.48914455692778913, 2.2856118046465235], [-3.3339080531067906, 2.13820144837754, -2.8768069331450343, 0.9810225418560473, -1.6826128969988041, -0.9367254081368033, -1.2065077041698287, 3.261183986707936, 0.7183984381357779, 3.167961831939942, -1.1502009131933266, -0.7747810734997123, 4.777106124879748, -5.01138419489596, -2.7044045679044326, -0.6044389063512237, 5.393168100594287, -1.9001985612965548, 2.309823359670628, 4.674965619350613, -0.7826236183679661, -0.5951252834627092, 4.938512503762813, -1.7363556839262682, -3.2359484791029667, 2.408889078057992, -1.625664561289168, 3.8165527566865665, 1.5406424869535023, -5.339729661996386, 0.7599740974154832, -0.04264154007832835, -2.067990000471, 3.2357455068342045, -5.393475526896024, -3.8400625728523945, -1.5702800847492067, 2.298642412979876, -4.708440463476941, -1.9983816639509104, -2.2353772864489283, -4.6274949398163425, -1.259587016256346, -3.55713255997161, -1.4669629486298192, 0.7460648837966826, -2.860235502876227, -2.7428479898888174, -3.0949534914881225, -2.6799164842745524], [-4.8444073973481165, -3.6791997427526284, 0.23157575873035557, -2.773063375051273, 9.70170569933119, -2.580039186391439, 1.2480696121792083, -4.65645110443284, -0.9377044577598113, -1.7737792576698381, -0.36490177982490235, 3.883621911581705, -0.6248200168448299, -3.7505988762862543, 0.25041013299823556, -3.0114447778933435, -0.4588671167841274, 4.719163588679504, 2.790828133100413, 3.2015361929850945, 1.5638997361579672, -0.09475297051666542, -4.67376515246313, -1.4539685801258335, -1.6220733355845416, 2.0282945400824524, 2.9835336909028696, 1.4355757119259676, -2.398753623227962, 5.551398486381418, 4.2595335924102615, -0.652942691426556, 0.5913437283247618, 1.9208396792224434, 0.3116739996326814, -0.2956067681401674, 1.4832592751825608, 0.03121828172830952, 0.6129340855600018, -4.83405046419924, -4.822767877263772, 5.223260137384247, -4.567108160814606, 1.461686481517834, 0.9611309081673572, 0.6675262752536205, -2.080814184628671, 3.0473331031573307, 1.0530544049350608, -1.0264833878123911]], "dtype": "float64"}, {"__nd__": [1.039323654687985, 1.1418533064926721, 0.795984676869268, 0.8627563353541181, 0.7331148444517519, 1.0629936280517978], "dtype": "float64"}, {"__nd__": [0, 10, 40, 1, 0, 7], "dtype": "int64"}, {"__nd__": [0.9, 1.0, 0.75, 1.0, 0.3, 0.95], "dtype": "float64"}, {"__nd__": [0.41261973772641336, 0.5468290320836933, 0.870899026484968, 0.632593993356869, 0.5196662199260424, 0.46265764374302243], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [12, 42, 41, 26, 16, 4], "dtype": "int64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.14045828217977369, 0.0, 0.0, 0.048824136089414376, 0.0, 0.0, 0.0, 0.5226162507054294, 0.0, 0.0, 0.0, 0.09948753463894768, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1886137963864348, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.07312279393079346, 0.0, 0.0, 0.023435646280220726, 0.0, 0.0, 0.0, 0.011880469595160245, 0.0, 0.0, 0.0, 0.0, 0.007113015555227622, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01085839737591891, 0.0, 0.0, 0.009389256654534384, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.037564398987325796, 0.0, 0.0, 0.16902675486054214, 0.0, 0.0, 0.0, 0.0, 0.2174369680386628, 0.0, 0.0, 0.44017229872161395, 0.0, 0.0, 0.0, 0.0], [0.0, 0.1391638344205781, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.09650500017291812, 0.41021434203153867, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1567454498361968, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.09257933597865593, 0.0, 0.0, 0.10479203756011238, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.9802442573132822, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01975574268671774, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[5.0, -5.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.001], "dtype": "float64"}, {"__nd__": [0], "dtype": "int64"}, {"__nd__": [1.0], "dtype": "float64"}, {"__nd__": [0.999999], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0], "dtype": "int64"}, {"__nd__": [[1.0, 0.0, 0.0]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(sample_batch, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")